# Mathematical Foundations: Demand Forecasting & Inventory Optimization

This notebook serves as the theoretical backbone of the project, formalizing the models and inventory policies used in the pipeline.

## 1. Problem Formulation

Let $Y_{i,t}$ denote the demand (sales) for SKU $i$ at time $t$. The goal is to predict $Y_{i, t+h}$ for a horizon $h=1, \dots, L$, where $L$ is the lead time, given the information set $\mathcal{F}_t = \{Y_{i, \tau}, X_{i, \tau} \mid \tau \le t\}$. $X$ represents exogenous variables like promotions and calendar events.

## 2. Modeling Approaches

### 2.1. Classical Approach: SARIMA

The Seasonal Autoregressive Integrated Moving Average (SARIMA) model provides a strong statistical baseline. It models the process as:

$$ \phi_p(B)\Phi_P(B^s)(1-B)^d(1-B^s)^D Y_t = \theta_q(B)\Theta_Q(B^s)\varepsilon_t $$

Where:
*   $B$ is the backshift operator ($B Y_t = Y_{t-1}$)
*   $\phi_p, \Phi_P$ are regular and seasonal AR polynomials.
*   $\theta_q, \Theta_Q$ are regular and seasonal MA polynomials.
*   $s$ is the seasonal period (e.g., $s=7$ for weekly seasonality).
*   $\varepsilon_t \sim WN(0, \sigma^2)$ is white noise.

### 2.2. Machine Learning Approach: Gradient Boosted Trees (LightGBM)

For large-scale, non-linear problems with many cross-sectional series, we use additive models of decision trees. We approximate the conditional expectation:

$$ \hat{Y}_{i,t} = \mathbb{E}[Y_{i,t} \mid \mathcal{F}_{t-1}] \approx \sum_{k=1}^K f_k(X_{i,t}, Y_{i,t-1}, Y_{i,t-7}, \dots) $$

Where $f_k \in \mathcal{F}$ are regression trees, minimizing a loss function $\mathcal{L}(y, \hat{y})$ (e.g., L1 loss for MAE, or pinball loss for quantiles).

## 3. Inventory Optimization Math

A forecast is only useful if it drives better decisions. We link the forecast error to inventory policies.

### 3.1. Forecast Error Variance over Lead Time

Let $\sigma_1^2$ be the variance of the 1-step ahead forecast error (empirical residual variance). Assuming errors are independent, the variance of the forecast error over a lead time $L$ is:

$$ \sigma_L^2 = L \cdot \sigma_1^2 \implies \sigma_L = \sigma_1 \sqrt{L} $$

### 3.2. Safety Stock

To achieve a Cycle Service Level (CSL) of $\alpha$ (e.g., 95%), we need to hold safety stock ($SS$) to cover demand uncertainty during the lead time. If lead time demand follows a normal distribution $N(\mu_L, \sigma_L^2)$:

$$ SS = z_\alpha \cdot \sigma_L $$

Where $z_\alpha = \Phi^{-1}(\alpha)$ is the inverse CDF of the standard normal distribution.

### 3.3. Order-Up-To Level (S)

The target inventory level to maintain stock availability is:

$$ S = \hat{Y}_L + SS = (\text{Forecast} \times L) + (z_\alpha \cdot \sigma_1 \sqrt{L}) $$

This formula directly shows how **reducing forecast error ($\sigma_1$) reduces the required safety stock**, translating directly to working capital savings.